# Finotello layer sweep

In [1]:
# Append path to deconversation modules
import sys
import os
import scanpy as sc
import numpy as np
import pandas as pd
#sys.path.append('../../deconversation')
sys.path.append("/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages")

/nfs/home/aoku/.local/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
import deconversation
from deconversation import embeddings as em
from deconversation import preprocessing as pr
from deconversation import deconvolution as de
from deconversation import visualization as vs 

geneformer successfully imported.
cell2sentence is not installed. Skipping related functions.
cellhermes is not installed. Skipping related functions.
scGPT is not installed. Skipping related functions.
scVI successfully imported.


In [3]:
dataset_name = 'finotello'
reference_path = '/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/hao800_signature_matrix_broad_id.csv'
bulk_path = '/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/finotello/finotello_tpm_id_transpose.csv'
ground_truth_path = '/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/finotello/finotello_facs.csv'

# Load signature
signature_df = pd.read_csv(reference_path, index_col=0)

# Load  bulk
bulk_df = pd.read_csv(bulk_path, index_col=0)

# Load ground truth
ground_truth = pd.read_csv(ground_truth_path, index_col=0)
ground_truth = ground_truth.T

In [4]:
def harmonize_columns(df):
    df = df.copy()
    if 'T cells CD8' in df.columns and 'T cells CD4 conv' in df.columns and 'Tregs' in df.columns:
        df['T cells'] = df['T cells CD8'] + df['T cells CD4 conv'] + df['Tregs']
        df = df.drop(columns=['T cells CD8', 'T cells CD4 conv', 'Tregs'])
    if 'pDC' in df.columns and 'mDC' in df.columns:
        df['mDC'] = df['mDC'] + df['pDC']
        df = df.drop(columns=['pDC'])
    elif 'pDC' in df.columns:
        df = df.rename(columns={'pDC': 'mDC'})
    return df

In [5]:
ground_truth = harmonize_columns(ground_truth)

In [6]:
ground_truth.head()

,NK cells,B cells,mDC,Monocytes,Neutrophils,Other,T cells
pbmc_1,0.0675,0.0581,0.0160,0.2001,0.0245,0.0802,0.5535
pbmc_2,0.1128,0.0296,0.0234,0.2481,0.0484,0.1044,0.4333
pbmc_3,0.1848,0.0420,0.0338,0.3180,0.0545,0.1031,0.2638
pbmc_4,0.1154,0.0384,0.0357,0.1943,0.0367,0.0999,0.4796
pbmc_5,0.0896,0.0606,0.0424,0.2636,0.0332,0.0508,0.4598


In [7]:
bulk_df.shape

(9, 18556)

### Zeroshot

In [18]:
layer_results = {}
metric_rows = []
celltype_rows = []
for layer in range(18, 19):
    print(f'=== layer {layer} ===', flush=True)
    sig_mat_gf_embed = em.extract_embs(
        bulk_df=signature_df,
        mode='geneformer',
        temp_output_dir='/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/',
        model_path='ctheodoris/Geneformer',
        delete_temp_files=True,
        layer_to_quant=layer,
    )
    gf_embed = em.extract_embs(
        bulk_df=bulk_df,
        mode='geneformer',
        temp_output_dir='/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/',
        model_path='ctheodoris/Geneformer',
        delete_temp_files=True,
        layer_to_quant=layer,
    )
    results = de.run_all_deconv(bulk_df=gf_embed.T, signature_df=sig_mat_gf_embed.T)
    results = {solver: harmonize_columns(df) for solver, df in results.items()}
    layer_results[layer] = results
    for solver, df in results.items():
        samples = df.index.intersection(ground_truth.index)
        celltypes = df.columns.intersection(ground_truth.columns)
        P = df.loc[samples, celltypes].astype(float)
        T = ground_truth.loc[samples, celltypes].astype(float)
        p = P.values.ravel()
        t = T.values.ravel()
        ok = np.isfinite(p) & np.isfinite(t)
        for ct in celltypes:
            a = P[ct].values
            b = T[ct].values
            m = np.isfinite(a) & np.isfinite(b)
            if m.sum() > 1 and np.std(a[m]) > 0 and np.std(b[m]) > 0:
                r_ct = np.corrcoef(a[m], b[m])[0, 1]
            else:
                r_ct = np.nan
            celltype_rows.append({'layer': layer, 'solver': solver, 'celltype': ct, 'correlation': r_ct, 'rmse': np.sqrt(np.mean((a[m] - b[m]) ** 2)) if m.sum() else np.nan})
        ct_sub = [r for r in celltype_rows if r['layer'] == layer and r['solver'] == solver]
        corrs = np.array([r['correlation'] for r in ct_sub], dtype=float)
        mean_corr = np.mean(np.nan_to_num(corrs, nan=0.0))
        mean_rmse = np.nanmean([r['rmse'] for r in ct_sub])
        metric_rows.append({'layer': layer, 'solver': solver, 'correlation': np.corrcoef(p[ok], t[ok])[0, 1], 'rmse': np.sqrt(np.mean((p[ok] - t[ok]) ** 2)), 'meanCorrelation': mean_corr, 'meanRMSE': mean_rmse})

metrics_df = pd.DataFrame(metric_rows)
celltype_df = pd.DataFrame(celltype_rows)

#metrics_df.to_csv('../../results/gf_layer_sweep/layer_sweep_metrics_zeroshot_finotello.csv', index=False)

=== layer 18 ===
Bulk AnnData saved to: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad
Starting Geneformer tokenization...
Tokenizing /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.57it/s]

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.



/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Serie

Loading Geneformer model...


...

Loading tokenized dataset...
Extracting Geneformer embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]

Condition number: 11.63.
~1-10: signatures are well separated; NNLS is usually hard to improve materially with another solver.
Smallest singular value: 0.1985.
If value is close to zero, the signature matrix is ill-conditioned; estimated proportions may be unstable.
Max pairwise cosine similarity between cell types: 0.9570.
Most similar pair: NK cells vs T cells.
If value is close to 1, these cell types are highly similar and difficult to resolve separately.
Running solver: nnls
Finished in 0.00 seconds.
Running solver: nnls_mod
Finished in 0.00 seconds.
Running solver: dwls
Finished in 0.01 seconds.
Running solver: simplex
Finished in 0.01 seconds.
Running solver: ridge_simplex
Finished in 0.01 seconds.
Running solver: dwls_simplex
Finished in 0.02 seconds.
Running solver: ridge
Finished in 0.01 seconds.
Running solver: elasticnet
Finished in 0.00 seconds.
Running solver: nusvr
Finished in 0.27 seconds.
Running solver: simplex_nnls
Finished in 0.01 seconds.
Running solver: gradient_de

/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:172: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  references = torch.tensor(references, dtype=torch.float32)
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mixture = torch.tensor(mixture, dtype=torch.float32)


Finished in 0.39 seconds.


In [19]:
sig_mat_gf_embed.to_csv("../../../ml_deconv_data/results_bulk/gf_embeds/gf_zs_finotello_embeddings.csv")

In [20]:
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# for ax, metric in zip(axes, ['correlation', 'rmse']):
#     for solver, sub in metrics_df.groupby('solver'):
#         sub = sub.sort_values('layer')
#         ax.plot(sub['layer'], sub[metric], marker='o', markersize=4, label=solver)
#     ax.set_xlabel('Geneformer layer (layer_to_quant)')
#     ax.set_ylabel(metric)
#     ax.set_title(metric)
#     ax.set_xticks(range(1, 19))
#     ax.grid(alpha=0.3, linestyle='--')
# axes[1].legend(title='Solver', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
# fig.tight_layout()
# #fig.savefig('../../results/gf_layer_sweep/layer_sweep_zeroshot_finotello.png', dpi=300, bbox_inches='tight')
# plt.show()
# print(metrics_df.loc[metrics_df.groupby('solver')['correlation'].idxmax()])

In [21]:
#visualize_solvers(layer_results[18], ground_truth, level=["cell_type"])

In [22]:
ground_truth.shape

(12, 7)

In [23]:
layer_results = {}
metric_rows = []
celltype_rows = []
for layer in range(18, 19):
    print(f'=== layer {layer} ===', flush=True)
    sig_mat_gf_embed = em.extract_embs(
        bulk_df=signature_df,
        mode='geneformer',
        temp_output_dir='/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/',
        model_path='/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/260624_geneformer_cellClassifier_gf_finetune/ksplit1/',
        delete_temp_files=True,
        layer_to_quant=layer,
    )
    gf_embed = em.extract_embs(
        bulk_df=bulk_df,
        mode='geneformer',
        temp_output_dir='/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/',
        model_path='/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/260624_geneformer_cellClassifier_gf_finetune/ksplit1/',
        delete_temp_files=True,
        layer_to_quant=layer,
    )
    results = de.run_all_deconv(bulk_df=gf_embed.T, signature_df=sig_mat_gf_embed.T)
    results = {solver: harmonize_columns(df) for solver, df in results.items()}
    layer_results[layer] = results
    for solver, df in results.items():
        samples = df.index.intersection(ground_truth.index)
        celltypes = df.columns.intersection(ground_truth.columns)
        P = df.loc[samples, celltypes].astype(float)
        T = ground_truth.loc[samples, celltypes].astype(float)
        p = P.values.ravel()
        t = T.values.ravel()
        ok = np.isfinite(p) & np.isfinite(t)
        for ct in celltypes:
            a = P[ct].values
            b = T[ct].values
            m = np.isfinite(a) & np.isfinite(b)
            if m.sum() > 1 and np.std(a[m]) > 0 and np.std(b[m]) > 0:
                r_ct = np.corrcoef(a[m], b[m])[0, 1]
            else:
                r_ct = np.nan
            celltype_rows.append({'layer': layer, 'solver': solver, 'celltype': ct, 'correlation': r_ct, 'rmse': np.sqrt(np.mean((a[m] - b[m]) ** 2)) if m.sum() else np.nan})
        ct_sub = [r for r in celltype_rows if r['layer'] == layer and r['solver'] == solver]
        corrs = np.array([r['correlation'] for r in ct_sub], dtype=float)
        mean_corr = np.mean(np.nan_to_num(corrs, nan=0.0))
        mean_rmse = np.nanmean([r['rmse'] for r in ct_sub])
        metric_rows.append({'layer': layer, 'solver': solver, 'correlation': np.corrcoef(p[ok], t[ok])[0, 1], 'rmse': np.sqrt(np.mean((p[ok] - t[ok]) ** 2)), 'meanCorrelation': mean_corr, 'meanRMSE': mean_rmse})

metrics_df = pd.DataFrame(metric_rows)
celltype_df = pd.DataFrame(celltype_rows)

#metrics_df.to_csv('../../results/gf_layer_sweep/layer_sweep_metrics_finetuned_finotello.csv', index=False)

=== layer 18 ===
Bulk AnnData saved to: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad
Starting Geneformer tokenization...
Tokenizing /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.58it/s]

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.



/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Serie

Loading Geneformer model...


...

Loading tokenized dataset...
Extracting Geneformer embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]

Condition number: 5.44.
~1-10: signatures are well separated; NNLS is usually hard to improve materially with another solver.
Smallest singular value: 0.3721.
If value is close to zero, the signature matrix is ill-conditioned; estimated proportions may be unstable.
Max pairwise cosine similarity between cell types: 0.8445.
Most similar pair: Monocytes vs mDC.
If value is close to 1, these cell types are highly similar and difficult to resolve separately.
Running solver: nnls
Finished in 0.00 seconds.
Running solver: nnls_mod
Finished in 0.00 seconds.
Running solver: dwls
Finished in 0.01 seconds.
Running solver: simplex
Finished in 0.01 seconds.
Running solver: ridge_simplex
Finished in 0.01 seconds.
Running solver: dwls_simplex
Finished in 0.02 seconds.
Running solver: ridge
Finished in 0.01 seconds.
Running solver: elasticnet
Finished in 0.00 seconds.
Running solver: nusvr
Finished in 0.33 seconds.
Running solver: simplex_nnls
Finished in 0.01 seconds.
Running solver: gradient_descen

/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:172: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  references = torch.tensor(references, dtype=torch.float32)
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mixture = torch.tensor(mixture, dtype=torch.float32)


Finished in 0.38 seconds.


In [24]:
sig_mat_gf_embed.to_csv("../../../ml_deconv_data/results_bulk/gf_embeds/gf_ft_finotello_embeddings.csv")

In [15]:
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# for ax, metric in zip(axes, ['correlation', 'rmse']):
#     for solver, sub in metrics_df.groupby('solver'):
#         sub = sub.sort_values('layer')
#         ax.plot(sub['layer'], sub[metric], marker='o', markersize=4, label=solver)
#     ax.set_xlabel('Geneformer layer (layer_to_quant)')
#     ax.set_ylabel(metric)
#     ax.set_title(metric)
#     ax.set_xticks(range(1, 19))
#     ax.grid(alpha=0.3, linestyle='--')
# axes[1].legend(title='Solver', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
# fig.tight_layout()
# fig.savefig('../../results/gf_layer_sweep/layer_sweep_finetuned_finotello.png', dpi=300, bbox_inches='tight')
# plt.show()
# print(metrics_df.loc[metrics_df.groupby('solver')['correlation'].idxmax()])

In [1]:
import pandas as pd

In [3]:
reference_path = '/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/hao800_signature_matrix_broad_symbol.csv'
bulk_path = '/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/finotello/finotello_tpm_symbol_transpose.csv'

In [10]:
# Signature matrix
signature_df = pd.read_csv(reference_path, index_col=0)
signature_df = signature_df.T
signature_df.columns.name = None
signature_df.index.name = None
signature_df = signature_df.T


# bulk data
bulk_df = pd.read_csv(bulk_path, index_col=0)
bulk_df = bulk_df.loc[bulk_df.index.dropna()]
bulk_df.columns.name = None
bulk_df.index.name = None

In [11]:
signature_df.head()

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A4GALT,AAAS,AACS,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
B cells,0.23625,0.02750,0.0,0.00125,0.00500,0.0,0.0,0.0,0.06000,0.02000,...,0.02875,0.03750,0.00375,0.03000,0.03625,0.09375,0.00250,0.11500,0.08750,0.17750
Monocytes,0.33750,0.03375,0.0,0.00250,0.00375,0.0,0.0,0.0,0.08125,0.02875,...,0.06625,0.03500,0.01000,0.01125,0.03000,0.16375,0.00000,0.32125,1.40125,0.32250
NK cells,0.10000,0.01250,0.0,0.01000,0.03875,0.0,0.0,0.0,0.05000,0.04125,...,0.03000,0.02375,0.00375,0.00750,0.02500,0.09750,0.00125,0.08750,0.39750,0.25875
T cells,0.25875,0.01750,0.0,0.00750,0.02625,0.0,0.0,0.0,0.05875,0.03000,...,0.02875,0.02875,0.01000,0.00875,0.02500,0.07500,0.00125,0.12125,0.28625,0.18875
mDC,0.37625,0.05000,0.0,0.01500,0.02750,0.0,0.0,0.0,0.12000,0.04625,...,0.11625,0.04250,0.01875,0.01500,0.03375,0.12875,0.00000,0.20375,1.07375,0.17250


In [12]:
bulk_df.head()

,UBE2Q2P2,SSX9,CXorf67,EFCAB8,SPATA31B1P,SDR16C6P,GTPBP6,EFCAB12,A1BG,A1CF,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,TPTEP1
pbmc_1,0.000000,0.000000,0.118865,0.000000,0,0.000000,57.280197,0.252188,10.863071,0.000000,...,3.477319,3.44931,6.36162,16.473360,0.426082,7.471390,327.737590,17.070511,14.530079,5.155380
pbmc_10,0.081115,0.000000,0.086782,0.000000,0,0.000000,72.275326,0.095742,14.785649,0.000000,...,3.712750,3.55546,5.50518,14.924620,0.222251,6.381709,442.838401,12.744583,9.246122,13.326484
pbmc_12,0.000000,0.000000,0.188464,0.031570,0,0.000000,50.997427,0.100608,7.492537,0.000000,...,2.937518,3.70517,7.47136,16.866000,0.409656,9.498991,364.599060,18.526458,13.210396,19.452058
pbmc_2,0.000000,0.000000,0.094006,0.004499,0,0.000000,60.604133,0.050183,15.827180,0.000000,...,2.454242,1.86107,3.35811,13.062773,0.405473,5.799880,219.376050,12.623465,5.652651,13.401427
pbmc_4,0.059691,0.212565,0.496705,0.346400,0,0.287881,104.405300,0.984864,10.540150,0.269025,...,2.885740,3.78000,9.81738,30.148749,4.520060,12.566580,250.918200,34.720530,12.323414,45.098055
